# 01 · Veri hazırlama ve veri denetimi (VS13 – kneeKL224)

Bu notebook şunları yapar:
1. Google Drive'ı bağlar ve proje klasörüne geçer
2. Gerekli ek kütüphaneleri kurar, ortam bilgisini kaydeder
3. `KneeXrayData.zip` arşivini indirir, SHA256 özetini hesaplar, **yalnızca `kneeKL224`** klasörünü oturum diskine açar
4. Resmi train/val/test dosya listelerini oluşturur (bölme değiştirilmez)
5. Veri denetimini çalıştırır: bütünlük, sınıf dağılımı, boyutlar, aykırı görüntüler, pHash yakın-kopya taraması, hasta çakışması

**Çalıştırma:** `Çalışma zamanı → Tümünü çalıştır`. Bu aşama GPU gerektirmez.

## 1. Drive ve proje klasörü

In [ ]:
import os, sys, glob, shutil, zipfile
ON_KAGGLE = os.path.exists('/kaggle/working')

if ON_KAGGLE:
    # ---- Kaggle: kod ve önceki çıktılar /kaggle/input altındaki veri setlerinden gelir (salt okunur) ----
    hits = glob.glob('/kaggle/input/**/src/config.py', recursive=True)
    assert hits, 'Kod veri seti bulunamadı: sağ panelden Add Input ile yzproje-code veri setini ekleyin.'
    code_root = os.path.dirname(os.path.dirname(hits[0]))
    PROJECT = '/kaggle/working/YZProje'
    shutil.copytree(code_root, PROJECT, dirs_exist_ok=True)          # kod her oturumda tazelenir
    os.makedirs(PROJECT + '/outputs', exist_ok=True)
    out_hits = glob.glob('/kaggle/input/**/splits/all_files.csv', recursive=True)
    assert out_hits, 'Çıktı veri seti bulunamadı: Add Input ile yzproje-outputs veri setini ekleyin.'
    out_src = os.path.dirname(os.path.dirname(out_hits[0]))
    # yalnızca çalışma klasöründe OLMAYAN dosyaları kopyala (oturum içi yeni sonuçlar ezilmesin)
    shutil.copytree(out_src, PROJECT + '/outputs', dirs_exist_ok=True,
                    copy_function=lambda s, d: None if os.path.exists(d) else shutil.copy2(s, d))
    os.makedirs('/kaggle/temp/data', exist_ok=True)
    os.environ['KNEE_DATA_ROOT'] = '/kaggle/temp/data'                # ham veri: geçici disk (çıktıya sayılmaz)
else:
    # ---- Colab: Drive ----
    from google.colab import drive
    drive.mount('/content/drive')
    # Projeyi Drive'da bul (iç içe klasör olsa bile); zip açılmamışsa aç
    hits = glob.glob('/content/drive/MyDrive/**/src/config.py', recursive=True)
    if not hits:
        zips = glob.glob('/content/drive/MyDrive/**/YZProje*.zip', recursive=True)
        if zips:
            zipfile.ZipFile(zips[0]).extractall('/content/drive/MyDrive/')
            hits = glob.glob('/content/drive/MyDrive/**/src/config.py', recursive=True)
    if not hits:
        raise FileNotFoundError('Drive içinde src/config.py bulunamadı.')
    roots = [os.path.dirname(os.path.dirname(h)) for h in hits]
    # Birden fazla kopya varsa, veri listeleri (outputs/splits) olan klasör tercih edilir
    with_out = [r for r in roots if os.path.exists(f'{r}/outputs/splits/all_files.csv')]
    PROJECT = sorted(with_out or roots, key=len)[0]
    os.environ['KNEE_DATA_ROOT'] = '/content/data'

os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = f'{PROJECT}/outputs'
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| Proje klasörü:', PROJECT)
print(sorted(os.listdir(PROJECT + '/outputs')) if os.path.exists(PROJECT + '/outputs') else 'outputs yok (ilk çalıştırma)')

## 2. Kurulum ve ortam bilgisi

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0
!mkdir -p outputs/env && pip freeze > outputs/env/pip_freeze.txt

from src.utils import env_info, save_json, set_seed
from src import config as C
set_seed(C.SEED)
info = env_info(); save_json(info, C.OUTPUT_DIR / 'env' / 'env_info.json'); info

## 3. İndirme ve çıkarma

`DATA_URL` değişkenine **YZ_2026_Proje_Veri_Seti_Atamalari.xlsx** dosyasındaki VS13 doğrudan indirme bağlantısını yapıştırın. Mendeley bağlantıları yönlendirme içerdiği için indirme `curl -L` ile yapılır ve kesilirse kaldığı yerden devam eder.

Arşivi daha önce Drive'a indirdiyseniz `ZIP_PATH` değişkenini doldurun, `DATA_URL` boş kalsın.

⏱ 7,17 GB indirme + SHA256 + çıkarma yaklaşık 10–20 dakika sürer. Colab oturumu kapanınca `/content/data` silinir; yeni oturumda bu hücre tekrar çalıştırılır (dosya listeleri Drive'da kalır).

In [ ]:
DATA_URL = ''      # ör. 'https://data.mendeley.com/public-files/datasets/56rmx5bjcr/files/.../file_downloaded'
ZIP_PATH = ''      # ör. '/content/drive/MyDrive/KneeXrayData.zip'

from pathlib import Path
from src import data_prep
if Path(C.EXTRACT_DIR).exists() and any(Path(C.EXTRACT_DIR).rglob('kneeKL224')):
    print('Veri zaten açılmış, yalnızca dosya listesi yenileniyor.')
    data_prep.main(['--skip-extract'])
elif DATA_URL:
    data_prep.main(['--url', DATA_URL])
elif ZIP_PATH:
    data_prep.main(['--zip', ZIP_PATH])
else:
    raise ValueError('DATA_URL veya ZIP_PATH doldurulmalı')

In [ ]:
# Ham arşiv artık gerekmiyorsa disk açmak için silinebilir (SHA256 özeti download_info.json'da kayıtlı)
# !rm -f /content/data/raw/KneeXrayData.zip
!df -h . | tail -1
print(open(C.AUDIT_DIR / 'download_info.json').read())

## 4. Veri denetimi

In [ ]:
from src import audit
audit.main([])

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(C.AUDIT_DIR / 'audit_summary.md', encoding='utf-8').read()))

## 5. Figürler

In [ ]:
for f in sorted(C.FIG_DIR.glob('audit_*.png')):
    print(f.name); display(Image(filename=str(f), width=900))

## 6. Sonraki adım

`outputs/audit/audit_summary.md`, `audit_summary.json` ve `outputs/figures/audit_*.png` dosyalarını değerlendirme için paylaşın. Dosya listeleri `outputs/splits/` altında sabitlendi; sonraki tüm deneyler bu listeleri kullanır.